# Cinderwright: Give Your AI Agent Access to 2,835 Paid APIs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cinderwright-ai/cinderwright-api/blob/main/examples/quickstart.ipynb)
[![PyPI](https://img.shields.io/pypi/v/cinderwright)](https://pypi.org/project/cinderwright/)

**Cinderwright** is a payment proxy for AI agents. Your agent calls a single endpoint, specifies a task in plain English, and the proxy finds the right service, pays for it (via Lightning or USDC), and returns the result.

Works with: **LangChain**, **LangGraph**, **CrewAI**, **OpenAI Agents SDK**, **smolagents**, **Pydantic AI**, or any Python code.

**No per-service API keys. No subscriptions. Pay per call.**

---

## What this notebook covers

1. Free demo -- zero setup, runs immediately
2. Direct usage with a Cinderwright account ($0.10 free credit)
3. LangChain / LangGraph agent integration
4. CrewAI agent integration
5. Funding your account with Bitcoin Lightning

In [ ]:
# Install Cinderwright
!pip install cinderwright -q
print('Installed.')

---
## Part 1: Free demo -- no account needed

Try it immediately. No API key, no wallet, no setup.

In [ ]:
import cinderwright

# Live Bitcoin price (calls CoinGecko via x402 micropayment, free demo)
print(cinderwright.demo('Bitcoin price'))

# Real-time weather
print(cinderwright.demo('weather in Tokyo'))

# General AI question
print(cinderwright.demo('explain x402 in one sentence'))

---
## Part 2: Get a free account ($0.10 credit, no deposit)

The free demo is limited. A Cinderwright account gives you access to all 2,835 services.

**To get a key**, run the cell below with any Ethereum/Base wallet address.
Don't have one? Use `0x0000000000000000000000000000000000000001` for testing.

In [ ]:
import requests

# Replace with your Base wallet address (or use the test address)
WALLET = '0x0000000000000000000000000000000000000001'

resp = requests.post(
    'https://api.ideafactorylab.org/proxy/setup',
    json={'wallet': WALLET}
)
data = resp.json()
API_KEY = data.get('key', '')
print('Your API key:', API_KEY)
print('Starting balance: $', data.get('balance_usd', 0.10))
print('\nSave this key -- you will use it in the cells below.')

In [ ]:
# Paste your key here if you already have one
# API_KEY = 'sk_cw_...'

# Verify it works
from cinderwright import Cinderwright

cw = Cinderwright(api_key=API_KEY)
print('Balance:', cw.balance())

---
## Part 3: Direct usage

Call any service with a plain English task. The proxy handles service discovery, protocol selection, and payment automatically.

In [ ]:
from cinderwright import Cinderwright

cw = Cinderwright(api_key=API_KEY)

# Each call finds the right service, pays for it, returns the result
tasks = [
    'Bitcoin price',
    'Ethereum price',
    'weather in London',
    'translate good morning to Japanese',
    'sentiment of: This product is absolutely amazing!',
]

for task in tasks:
    result = cw.ask(task)
    print(f'Task: {task}')
    print(f'Result: {result}')
    print()

print('Remaining balance:', cw.balance()['balance_usd'])

In [ ]:
# Search the service index
services = cw.search('translation')
print(f'Found {len(services)} translation services')
for s in services[:3]:
    print(f"  {s.get('name')} -- {s.get('url')}")

---
## Part 4: LangChain / LangGraph

Drop `CinderwrightTool` into any LangChain agent as a single tool that covers hundreds of services.

In [ ]:
!pip install langchain-core langchain-openai langgraph -q
print('LangChain installed.')

In [ ]:
import os

# Set your OpenAI key -- or use Gemini (see commented section below)
# os.environ['OPENAI_API_KEY'] = 'sk-...'

# --- Gemini (free tier, no credit card) ---
# Get key at: https://aistudio.google.com/apikey
# !pip install langchain-google-genai -q
# os.environ['GOOGLE_API_KEY'] = 'AIza...'

from cinderwright.langchain import CinderwrightTool

# One tool, 2835 services
cw_tool = CinderwrightTool(api_key=API_KEY)

print('Tool name:', cw_tool.name)
print('Tool ready. Test it directly:')
print(cw_tool.invoke({'task': 'Bitcoin price'}))

In [ ]:
# Full LangChain agent (requires OpenAI or Gemini key above)
from langchain_openai import ChatOpenAI  # swap for ChatGoogleGenerativeAI if using Gemini
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
# llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash')  # Gemini option

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant with access to real-time data and services.'),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, [cw_tool], prompt)
executor = AgentExecutor(agent=agent, tools=[cw_tool], verbose=True)

result = executor.invoke({
    'input': 'What is the Bitcoin price today, and how is the weather in Tokyo?'
})
print('\nFinal answer:', result['output'])

In [ ]:
# LangGraph (minimal, no LLM key needed for this example)
# Shows how to use the tool in a LangGraph node directly

from langgraph.graph import StateGraph, END
from typing import TypedDict

class State(TypedDict):
    task: str
    result: str

def cinderwright_node(state: State) -> State:
    """LangGraph node that calls a Cinderwright service."""
    result = cw_tool.invoke({'task': state['task']})
    return {'task': state['task'], 'result': result}

graph = StateGraph(State)
graph.add_node('call_service', cinderwright_node)
graph.set_entry_point('call_service')
graph.add_edge('call_service', END)
app = graph.compile()

output = app.invoke({'task': 'Bitcoin price', 'result': ''})
print('LangGraph result:', output['result'])

---
## Part 5: CrewAI

Add Cinderwright as a tool to any CrewAI agent.

In [ ]:
!pip install crewai -q
print('CrewAI installed.')

In [ ]:
from cinderwright.crewai import CinderwrightTool as CrewCWTool
from crewai import Agent, Task, Crew, LLM

cw_crew_tool = CrewCWTool(api_key=API_KEY)

# Test the tool directly (no LLM needed)
print('Direct tool call:', cw_crew_tool.run('Ethereum price'))
print()

# Full CrewAI agent (requires LLM API key)
# llm = LLM(model='gpt-4o-mini', api_key='sk-...')
# researcher = Agent(
#     role='Market Analyst',
#     goal='Provide accurate market data',
#     backstory='Expert at gathering and analyzing real-time financial data',
#     tools=[cw_crew_tool],
#     llm=llm,
#     verbose=True,
# )
# task = Task(
#     description='Get the current price of Bitcoin and Ethereum',
#     agent=researcher,
#     expected_output='Current prices for BTC and ETH in USD',
# )
# crew = Crew(agents=[researcher], tasks=[task], verbose=True)
# print(crew.kickoff())

---
## Part 6: Fund your account with Bitcoin Lightning

No USDC? No problem. Pay a Lightning invoice and your account is credited within 30 seconds.

In [ ]:
from cinderwright import Cinderwright
import time

cw = Cinderwright(api_key=API_KEY)

# Create a Lightning deposit invoice
invoice = cw.deposit_lightning(amount_sats=1000)  # 1000 sats ~ $0.60

print('Pay this Lightning invoice from any Bitcoin wallet:')
print()
print(invoice['payment_request'])
print()
print(f"Amount: {invoice['amount_sats']} sats (${invoice['amount_usd']:.4f})")
print(f"Expires: {invoice['expires_at']}")
print()
print('After paying, run the cell below to check your balance.')

In [ ]:
# Check if payment was received (runs for up to 90 seconds)
r_hash = invoice['r_hash']

for i in range(9):
    time.sleep(10)
    status = cw.check_deposit(r_hash)
    print(f'Check {i+1}/9: {status["state"]}')
    if status['settled']:
        print(f'Credited! New balance: ${cw.balance()["balance_usd"]}')
        break
else:
    print('Not settled yet -- payment may still arrive. Check again in a minute.')

---
## Next steps

- **Browse services**: `cw.search('your category')` or [api.ideafactorylab.org/discover](https://api.ideafactorylab.org/discover)
- **MCP server** (Claude integration): `npx cinderwright-mcp-server`
- **Node.js CLI**: `npx cinderwright "Bitcoin price"`
- **Source + docs**: [github.com/cinderwright-ai/cinderwright-api](https://github.com/cinderwright-ai/cinderwright-api)
- **OpenAI Agents SDK**: `from cinderwright.openai_agents import make_cinderwright_tool`
- **smolagents**: `from cinderwright.tools import CinderwrightSmolTool`

Questions or issues: open a GitHub issue or find us on Reddit (u/cinderwright-ai).